# Ex 1.1 drift/diffusion slices and roll-outs

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Appendix J (Figures 6–7), Appendix K (Figure 8), Appendix L.

Runs the 1D pipeline on Ex 1.1 and produces the drift/diffusion slice comparisons, paired roll-outs under shared noise realizations, and the learned-generator coefficient summary. All figures and metrics are saved to a results folder and zipped.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Config, imports, global knobs

In [ ]:
# ============================================================
# Cell 1 — Config, imports, global knobs
# ============================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import math
import time
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Result saving / automatic notebook logging
# ============================================================

import os
import sys
import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime

# Root directory for every saved result from this run.
# Relative to the current working directory.
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

RESULTS_DIR = Path.cwd() / f"sde_surrogate_results_{RUN_TIMESTAMP}"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
LOGS_DIR = RESULTS_DIR / "logs"

for d in [RESULTS_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PRINT_LOG_PATH = LOGS_DIR / f"notebook_print_log_{RUN_TIMESTAMP}.txt"


class Tee:
    """
    Duplicate stdout/stderr to both notebook output and a log file.
    Add this before any print statements if you want the whole run captured.
    """
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for s in self.streams:
            s.write(data)
            s.flush()

    def flush(self):
        for s in self.streams:
            s.flush()


_ORIGINAL_STDOUT = sys.stdout
_ORIGINAL_STDERR = sys.stderr

_log_file = open(PRINT_LOG_PATH, "w", encoding="utf-8")
sys.stdout = Tee(_ORIGINAL_STDOUT, _log_file)
sys.stderr = Tee(_ORIGINAL_STDERR, _log_file)


# ------------------------------------------------------------
# Automatically save every matplotlib figure when plt.show() runs
# ------------------------------------------------------------

_ORIGINAL_PLT_SHOW = plt.show
_SAVED_FIGURE_IDS = set()
_FIGURE_COUNTER = {"n": 0}


def _sanitize_filename(s, max_len=80):
    s = str(s).strip()
    if not s:
        return "untitled"
    keep = []
    for ch in s:
        if ch.isalnum() or ch in ("-", "_", "."):
            keep.append(ch)
        elif ch.isspace():
            keep.append("_")
        else:
            keep.append("_")
    out = "".join(keep)
    while "__" in out:
        out = out.replace("__", "_")
    return out[:max_len].strip("_") or "untitled"


def _get_figure_title(fig):
    if fig._suptitle is not None:
        return fig._suptitle.get_text()

    titles = []
    for ax in fig.axes:
        title = ax.get_title()
        if title:
            titles.append(title)

    if titles:
        return titles[0]

    return "figure"


def _save_new_open_figures():
    """
    Save every currently open matplotlib figure that has not already been saved.
    Existing plt.show() calls throughout the notebook will trigger this.
    """
    for num in plt.get_fignums():
        fig = plt.figure(num)
        fig_id = id(fig)

        if fig_id in _SAVED_FIGURE_IDS:
            continue

        _FIGURE_COUNTER["n"] += 1
        title = _sanitize_filename(_get_figure_title(fig))
        stem = f"{_FIGURE_COUNTER['n']:03d}_{title}"

        png_path = FIGURES_DIR / f"{stem}.png"
        pdf_path = FIGURES_DIR / f"{stem}.pdf"

        try:
            fig.tight_layout()
        except Exception:
            pass

        fig.savefig(png_path, dpi=200, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")

        _SAVED_FIGURE_IDS.add(fig_id)
        print(f"[Saved figure] {png_path}")
        print(f"[Saved figure] {pdf_path}")


def show_and_save_all_figures(*args, **kwargs):
    _save_new_open_figures()
    return _ORIGINAL_PLT_SHOW(*args, **kwargs)


plt.show = show_and_save_all_figures

print(f"Saving all results to: {RESULTS_DIR}")
print(f"Saving print log to: {PRINT_LOG_PATH}")

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


# ----------------------------
# Main configuration knobs
# ----------------------------
CONFIG = {
    # Reproducibility
    "SEED": 42,

    # Synthetic data generation
    "T_FINAL": 2.0,
    "DT": 0.01,
    "N_TRAJECTORIES": 96,
    "X0_MEAN": 0.0,
    "X0_STD": 0.7,
    "TRAIN_FRACTION": 0.8,

    # Neural SDE surrogate
    "MLP_HIDDEN_DIM": 64,
    "MLP_DEPTH": 3,
    "INPUT_T_SCALE": 2.0,     # t is mapped roughly to [-1,1] by t / INPUT_T_SCALE
    "INPUT_X_SCALE": 2.5,     # x is mapped by x / INPUT_X_SCALE
    "SIGMA_EPS": 1e-4,

    # Baseline surrogate training
    "BASELINE_EPOCHS": 1500,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-3,
    "PRINT_EVERY": 250,

    # Symmetry discovery
    "N_GENERATORS": 3,              # try 1, 2, or 3 in 1D
    "SYM_DISCOVERY_EPOCHS": 2000,
    "SYM_DISCOVERY_LR": 1e-2,
    "SYM_DISCOVERY_POINTS": 512,
    "SYM_DISCOVERY_T_BOX": (0.0, 2.0),
    "SYM_DISCOVERY_X_BOX": (-2.5, 2.5),
    "SYM_INIT_SCALE": 0.05,

    # Symmetry-discovery loss weights
    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.1,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.15,
    "CLOSURE_RCOND": 1e-5,

    # Extra Gaeta Example 8 generic linear-drift basis elements.
    # Example 8 corresponds to linear drift f(x)=mu*x with constant diffusion.
    # The paper explicitly discusses linear drift with constant sigma; in 1D,
    # the corresponding family gives e^{mu t}∂x and e^{2mu t}(∂t + mu x∂x).
    "BASIS_LINEAR_MU_LIST": [-2.0, -0.5, 0.5, 2.0],

    # Exponential clipping for numerical safety in basis functions
    "BASIS_EXP_CLIP": 8.0,

    # Stage F: symmetry-regularized surrogate refinement
    "STAGE_F_EPOCHS": 1200,
    "STAGE_F_BATCH_SIZE": 512,
    "STAGE_F_LR": 1e-3,
    "STAGE_F_LAMBDA_SYM_DET": 0.05,
    "STAGE_F_LAMBDA_PROJECTABLE": 0.01,

    # Extra collocation points for Stage F symmetry loss
    # These are in addition to the trajectory data points in each batch.
    "STAGE_F_N_EXTRA_SYM_POINTS": 1024,
    "STAGE_F_EXTRA_T_BOX": (0.0, 2.0),
    "STAGE_F_EXTRA_X_BOX": (-2.5, 2.5),

    # Diagnostics
    "N_PLOT_GRID": 200,

    # Roll-out visualization after baseline and Stage F training
    "ROLLOUT_VIS_N_PATHS": 48,
    "ROLLOUT_VIS_T_BOX": (0.0, 2.0),
    "ROLLOUT_VIS_X_BOX": (-2.5, 2.5),
    "ROLLOUT_VIS_X0_BOX": (-1.0, 1.0),
    "ROLLOUT_VIS_SEED": 2025,
}

key = jax.random.PRNGKey(CONFIG["SEED"])

In [ ]:
def rmse(a, b):
    return jnp.sqrt(jnp.mean((a - b) ** 2))


def mae(a, b):
    return jnp.mean(jnp.abs(a - b))


def mse(a, b):
    return jnp.mean((a - b) ** 2)


def safe_percent_change(new, old, eps=1e-12):
    """
    Percent change from old to new.
    Negative means the metric decreased.
    """
    return 100.0 * (new - old) / (abs(old) + eps)


def safe_percent_reduction(new, old, eps=1e-12):
    """
    Percent reduction from old to new.
    Positive means the metric improved/decreased.
    """
    return 100.0 * (old - new) / (abs(old) + eps)


def relative_l2_error(pred, truth, eps=1e-12):
    return jnp.sqrt(jnp.sum((pred - truth) ** 2) / (jnp.sum(truth ** 2) + eps))

## Cell 2 — Synthetic non-Gaeta SDE data

In [ ]:
# ============================================================
# Cell 2 — Synthetic non-Gaeta SDE data
# ============================================================

def true_drift(t, x):
    """
    Synthetic drift not taken from Gaeta's canonical examples.
    """
    return -0.8 * x + 0.25 * jnp.sin(1.7 * t) + 0.1 * jnp.tanh(x)


def true_sigma(t, x):
    """
    Synthetic positive diffusion not taken from Gaeta's canonical examples.
    """
    return 0.35 + 0.08 * (jnp.sin(t) ** 2) + 0.12 * jax.nn.sigmoid(x)


@jax.jit
def simulate_sde_paths(key):
    """
    Simulate all trajectories using Euler-Maruyama.

    Returns
    -------
    times : shape (n_steps + 1,)
    paths : shape (n_paths, n_steps + 1)
    """
    T = CONFIG["T_FINAL"]
    dt = CONFIG["DT"]
    n_steps = int(round(T / dt))
    n_paths = CONFIG["N_TRAJECTORIES"]

    key_x0, key_noise = jax.random.split(key)
    x0 = CONFIG["X0_MEAN"] + CONFIG["X0_STD"] * jax.random.normal(key_x0, (n_paths,))
    eps = jax.random.normal(key_noise, (n_steps, n_paths))

    times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, noise = inp
        f = true_drift(t, x)
        s = true_sigma(t, x)
        x_next = x + f * dt + s * sqrt_dt * noise
        return x_next, x_next

    _, xs_tail = jax.lax.scan(step, x0, (times[:-1], eps))
    paths = jnp.concatenate([x0[None, :], xs_tail], axis=0).T
    return times, paths


def make_increment_dataset(times, paths):
    """
    Convert simulated trajectories into one-step increment data.

    Returns tuple:
        t0, x0, dt, dx
    each shape (n_samples,)
    """
    n_paths, n_times = paths.shape
    n_steps = n_times - 1
    dt = CONFIG["DT"]

    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1].reshape(-1)
    dx = (paths[:, 1:] - paths[:, :-1]).reshape(-1)
    dt_arr = jnp.full_like(dx, dt)

    return t0, x0, dt_arr, dx


def train_test_split_dataset(key, dataset, train_fraction):
    n = dataset[0].shape[0]
    perm = jax.random.permutation(key, n)
    n_train = int(train_fraction * n)

    idx_train = perm[:n_train]
    idx_test = perm[n_train:]

    train = tuple(arr[idx_train] for arr in dataset)
    test = tuple(arr[idx_test] for arr in dataset)
    return train, test


def sample_batch(key, dataset, batch_size):
    n = dataset[0].shape[0]
    idx = jax.random.randint(key, (batch_size,), 0, n)
    return tuple(arr[idx] for arr in dataset)


key, subkey = jax.random.split(key)
times, paths = simulate_sde_paths(subkey)
dataset = make_increment_dataset(times, paths)

key, subkey = jax.random.split(key)
train_data, test_data = train_test_split_dataset(
    subkey, dataset, CONFIG["TRAIN_FRACTION"]
)

print("times shape:", times.shape)
print("paths shape:", paths.shape)
print("train samples:", train_data[0].shape[0])
print("test samples:", test_data[0].shape[0])

plt.figure(figsize=(8, 4))
for i in range(min(12, CONFIG["N_TRAJECTORIES"])):
    plt.plot(np.array(times), np.array(paths[i]), alpha=0.6)
plt.xlabel("t")
plt.ylabel("X_t")
plt.title("Synthetic sparse SDE trajectories")
plt.grid(True)
plt.show()

## Cell 3 — Neural SDE surrogate, Adam optimizer, baseline training

In [ ]:
# ============================================================
# Cell 3 — Neural SDE surrogate, Adam optimizer, baseline training
# ============================================================

def glorot_uniform(key, in_dim, out_dim):
    lim = jnp.sqrt(6.0 / (in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-lim, maxval=lim)


class MLP:
    """
    Small pure-JAX MLP class.

    Output dimension is 2:
        raw[0] -> drift f_theta(t,x)
        raw[1] -> raw diffusion, transformed by softplus
    """

    def __init__(self, in_dim=2, hidden_dim=64, depth=3, out_dim=2):
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.depth = depth
        self.out_dim = out_dim

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)

        params = []
        for k, in_d, out_d in zip(keys, sizes[:-1], sizes[1:]):
            W = glorot_uniform(k, in_d, out_d)
            b = jnp.zeros((out_d,), dtype=jnp.float32)
            params.append({"W": W, "b": b})
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        out = h @ params[-1]["W"] + params[-1]["b"]
        return out


surrogate_model = MLP(
    in_dim=2,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2,
)


def normalize_input(t, x):
    tn = t / CONFIG["INPUT_T_SCALE"]
    xn = x / CONFIG["INPUT_X_SCALE"]
    return jnp.array([tn, xn], dtype=jnp.float32)


def surrogate_f_sigma(params, t, x):
    z = normalize_input(t, x)
    raw = surrogate_model.apply(params, z)

    f = raw[0]
    sigma = jax.nn.softplus(raw[1]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def f_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1])[0]


def sigma_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1])[1]


def data_nll_loss(params, batch):
    """
    Euler-Maruyama Gaussian negative log-likelihood.
    batch = (t0, x0, dt, dx)
    """
    t0, x0, dt, dx = batch

    f, sigma = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)
    mean = f * dt
    var = sigma**2 * dt + 1e-8

    nll = 0.5 * ((dx - mean) ** 2 / var + jnp.log(2.0 * jnp.pi * var))
    return jnp.mean(nll)


# ----------------------------
# Lightweight pure-JAX Adam
# ----------------------------
def adam_init(params):
    m = jax.tree_util.tree_map(jnp.zeros_like, params)
    v = jax.tree_util.tree_map(jnp.zeros_like, params)
    t = jnp.array(0, dtype=jnp.int32)
    return {"m": m, "v": v, "t": t}


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1

    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"],
        grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * (g * g),
        state["v"],
        grads,
    )

    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)

    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - lr * m / (jnp.sqrt(v) + eps),
        params,
        m_hat,
        v_hat,
    )

    return params_new, {"m": m_new, "v": v_new, "t": t_new}


@jax.jit
def baseline_train_step(params, opt_state, batch):
    loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
    params, opt_state = adam_update(
        params, grads, opt_state, CONFIG["BASELINE_LR"]
    )
    return params, opt_state, loss


# ----------------------------
# Train baseline surrogate
# ----------------------------
key, subkey = jax.random.split(key)
sde_params = surrogate_model.init(subkey)
sde_opt_state = adam_init(sde_params)

baseline_losses = []
t_start = time.time()

for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    batch = sample_batch(subkey, train_data, CONFIG["BASELINE_BATCH_SIZE"])

    sde_params, sde_opt_state, loss = baseline_train_step(
        sde_params, sde_opt_state, batch
    )

    baseline_losses.append(float(loss))

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        train_loss = float(data_nll_loss(sde_params, train_data))
        test_loss = float(data_nll_loss(sde_params, test_data))
        print(
            f"[Baseline] epoch {epoch:5d} | "
            f"batch nll {float(loss): .5f} | "
            f"train nll {train_loss: .5f} | "
            f"test nll {test_loss: .5f}"
        )

print(f"Baseline training time: {time.time() - t_start:.2f} sec")

## Cell 4 — Editable Gaeta SDE symmetry-basis library

In [ ]:
# ============================================================
# Cell 4 — Editable Gaeta SDE symmetry-basis library
# ============================================================

def safe_exp(z):
    c = CONFIG["BASIS_EXP_CLIP"]
    return jnp.exp(jnp.clip(z, -c, c))


# ------------------------------------------------------------------
# Editable symmetry basis library
#
# Each basis function returns a 2-vector:
#     [tau(t), xi(t,x)]
#
# The learned generator is:
#     X_a(t,x) = sum_r W[a,r] B_r(t,x)
#
# W[a,r] are trainable scalar constants, not functions of (t,x).
#
# The library below includes the 1D-compatible SDE-level generators
# derived or implied by Gaeta's examples.
#
# 2D examples such as Kramers/Finkel-type systems are not directly
# inserted into this 1D notebook, because their generators act on
# more than one spatial coordinate.
# ------------------------------------------------------------------

BASIS_LIBRARY = []


def add_basis(name, source, fn):
    BASIS_LIBRARY.append({"name": name, "source": source, "fn": fn})


# ----------------------------
# Example 1: Brownian motion dx = sigma0 dW
# SDE symmetries: ∂t, ∂x, 2t∂t + x∂x
# ----------------------------
add_basis(
    "E1_time_translation__dt",
    "Gaeta Example 1: Brownian SDE",
    lambda t, x: jnp.array([1.0, 0.0], dtype=jnp.float32),
)

add_basis(
    "E1_space_translation__dx",
    "Gaeta Example 1: Brownian SDE",
    lambda t, x: jnp.array([0.0, 1.0], dtype=jnp.float32),
)

add_basis(
    "E1_brownian_scaling__2t_dt_plus_x_dx",
    "Gaeta Example 1: Brownian SDE",
    lambda t, x: jnp.array([2.0 * t, x], dtype=jnp.float32),
)


# ----------------------------
# Example 2: dx = dt + x dW
# SDE symmetry is only time translation ∂t.
# Already included above, but retained conceptually in comments.
# ----------------------------


# ----------------------------
# Example 3: dx = x dt + dW
# SDE symmetries:
#   ∂t,
#   e^t ∂x,
#   e^{2t}(∂t + x∂x)
# ----------------------------
add_basis(
    "E3_exp_t_space__e_t_dx",
    "Gaeta Example 3: dx = x dt + dW",
    lambda t, x: jnp.array([0.0, safe_exp(t)], dtype=jnp.float32),
)

add_basis(
    "E3_exp_2t_projective__e2t_dt_plus_e2t_x_dx",
    "Gaeta Example 3: dx = x dt + dW",
    lambda t, x: jnp.array([safe_exp(2.0 * t), safe_exp(2.0 * t) * x], dtype=jnp.float32),
)


# ----------------------------
# Example 7: 1D version of n uncoupled Langevin oscillators
# dx = -x dt + sqrt(2s) dW
# SDE symmetries:
#   ∂t,
#   e^{-t} ∂x,
#   e^{-2t}(∂t - x∂x)
# ----------------------------
add_basis(
    "E7_exp_minus_t_space__em_t_dx",
    "Gaeta Example 7: dx = -x dt + const dW",
    lambda t, x: jnp.array([0.0, safe_exp(-t)], dtype=jnp.float32),
)

add_basis(
    "E7_exp_minus_2t_projective__em2t_dt_minus_em2t_x_dx",
    "Gaeta Example 7: dx = -x dt + const dW",
    lambda t, x: jnp.array([safe_exp(-2.0 * t), -safe_exp(-2.0 * t) * x], dtype=jnp.float32),
)


# ----------------------------
# Example 8: linear drift f(x)=mu*x, constant diffusion.
# 1D-compatible family:
#   e^{mu t} ∂x,
#   e^{2mu t}(∂t + mu x∂x)
#
# The explicit mu=+1 and mu=-1 cases above correspond to Examples 3 and 7.
# Here we add extra user-configurable mu values to enlarge the editable dictionary.
# ----------------------------
def make_linear_mu_space(mu):
    return lambda t, x: jnp.array([0.0, safe_exp(mu * t)], dtype=jnp.float32)


def make_linear_mu_projective(mu):
    return lambda t, x: jnp.array(
        [safe_exp(2.0 * mu * t), mu * safe_exp(2.0 * mu * t) * x],
        dtype=jnp.float32,
    )


for mu in CONFIG["BASIS_LINEAR_MU_LIST"]:
    if abs(mu) > 1e-8:
        add_basis(
            f"E8_linear_mu_{mu:+.2f}__e_mut_dx",
            "Gaeta Example 8: linear drift constant diffusion, 1D family",
            make_linear_mu_space(float(mu)),
        )
        add_basis(
            f"E8_linear_mu_{mu:+.2f}__e_2mut_dt_plus_mu_x_dx",
            "Gaeta Example 8: linear drift constant diffusion, 1D family",
            make_linear_mu_projective(float(mu)),
        )


# ----------------------------
# Example 10: nonlinear radial system.
# In 1D with lambda != 0, only ∂t survives.
# Already included as E1_time_translation__dt.
# ----------------------------


BASIS_NAMES = [b["name"] for b in BASIS_LIBRARY]
BASIS_SOURCES = [b["source"] for b in BASIS_LIBRARY]
BASIS_FNS = tuple(b["fn"] for b in BASIS_LIBRARY)
N_BASIS = len(BASIS_FNS)

print(f"Number of basis functions: {N_BASIS}")
for i, (nm, src) in enumerate(zip(BASIS_NAMES, BASIS_SOURCES)):
    print(f"{i:02d} | {nm} | {src}")


def eval_basis_one(t, x):
    """
    Returns basis values at one point.

    Output shape:
        (R, 2)
    where component 0 is tau and component 1 is xi.
    """
    return jnp.stack([fn(t, x) for fn in BASIS_FNS], axis=0)


def eval_generators_point(W, z):
    """
    Evaluate learned generators at one point.

    W shape:
        (m, R)

    z shape:
        (2,) = [t, x]

    Returns:
        X shape (m, 2), with X[a] = [tau_a, xi_a].
    """
    t, x = z[0], z[1]
    B = eval_basis_one(t, x)       # (R, 2)
    X = W @ B                      # (m, 2)
    return X


def eval_generators_batch(W, tx):
    """
    tx shape:
        (S, 2)

    Returns:
        X shape (S, m, 2)
    """
    return jax.vmap(lambda z: eval_generators_point(W, z))(tx)


def sample_tx_box(key, n_points, t_box, x_box):
    key_t, key_x = jax.random.split(key)
    t = jax.random.uniform(
        key_t,
        (n_points,),
        minval=float(t_box[0]),
        maxval=float(t_box[1]),
    )
    x = jax.random.uniform(
        key_x,
        (n_points,),
        minval=float(x_box[0]),
        maxval=float(x_box[1]),
    )
    return jnp.stack([t, x], axis=1)


key, subkey = jax.random.split(key)
W = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
    subkey,
    (CONFIG["N_GENERATORS"], N_BASIS),
)

print("Initial W shape:", W.shape)

## Cell 5 — Gaeta determining loss, global Lie closure, symmetry discovery

In [ ]:
# ============================================================
# Cell 5 — Gaeta determining loss, global Lie closure, symmetry discovery
# ============================================================

def det_residuals_point(sde_params, W, z):
    """
    1D specialization of Gaeta's projectable Ito SDE determining equations.

    Candidate generator:
        X_a = tau_a(t) ∂t + xi_a(t,x) ∂x

    Neural SDE:
        dX_t = f_theta(t,X_t) dt + sigma_theta(t,X_t) dW_t

    Residuals:
        R_f =
            xi_t + f xi_x - xi f_x - ∂t(f tau)
            + 0.5 sigma^2 xi_xx

        R_sigma =
            sigma xi_x - xi sigma_x - tau sigma_t
            - 0.5 sigma tau_t

    Returns:
        R_f shape (m,)
        R_sigma shape (m,)
        tau_x shape (m,)
    """
    f = f_single(sde_params, z)
    sigma = sigma_single(sde_params, z)

    grad_f = jax.grad(lambda zz: f_single(sde_params, zz))(z)
    grad_sigma = jax.grad(lambda zz: sigma_single(sde_params, zz))(z)

    f_t, f_x = grad_f[0], grad_f[1]
    sigma_t, sigma_x = grad_sigma[0], grad_sigma[1]

    gen_fun = lambda zz: eval_generators_point(W, zz)
    X = gen_fun(z)                              # (m, 2)
    J = jax.jacfwd(gen_fun)(z)                  # (m, 2, 2)
    H = jax.jacfwd(jax.jacfwd(gen_fun))(z)      # (m, 2, 2, 2)

    tau = X[:, 0]
    xi = X[:, 1]

    tau_t = J[:, 0, 0]
    tau_x = J[:, 0, 1]
    xi_t = J[:, 1, 0]
    xi_x = J[:, 1, 1]
    xi_xx = H[:, 1, 1, 1]

    R_f = (
        xi_t
        + f * xi_x
        - xi * f_x
        - (f_t * tau + f * tau_t)
        + 0.5 * sigma**2 * xi_xx
    )

    R_sigma = (
        sigma * xi_x
        - xi * sigma_x
        - tau * sigma_t
        - 0.5 * sigma * tau_t
    )

    return R_f, R_sigma, tau_x


def determining_loss(sde_params, W, tx):
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, W, z)
    )(tx)

    det = jnp.mean(R_f**2 + R_sigma**2)
    projectable = jnp.mean(tau_x**2)
    return det, projectable


def generator_jacobians_batch(W, tx):
    """
    Returns:
        X shape (S, m, 2)
        J shape (S, m, 2, 2)
    """
    def one(z):
        gen_fun = lambda zz: eval_generators_point(W, zz)
        X = gen_fun(z)
        J = jax.jacfwd(gen_fun)(z)
        return X, J

    return jax.vmap(one)(tx)


def lie_closure_loss_global(W, tx):
    """
    Global-batch Lie closure.

    We solve one least-squares problem over the whole collocation batch:

        [X_i, X_j] ≈ sum_k C_ij^k X_k

    This makes the structure constants global by construction.
    Therefore we do NOT need a separate structure-constant constancy loss.
    """
    m = W.shape[0]
    if m < 2:
        return jnp.array(0.0, dtype=jnp.float32)

    X, J = generator_jacobians_batch(W, tx)
    S = tx.shape[0]

    idx_i, idx_j = jnp.triu_indices(m, k=1)
    P = idx_i.shape[0]

    Xi = X[:, idx_i, :]     # (S, P, 2)
    Xj = X[:, idx_j, :]     # (S, P, 2)
    Ji = J[:, idx_i, :, :]  # (S, P, 2, 2)
    Jj = J[:, idx_j, :, :]  # (S, P, 2, 2)

    tau_i, xi_i = Xi[..., 0], Xi[..., 1]
    tau_j, xi_j = Xj[..., 0], Xj[..., 1]

    tau_i_t = Ji[..., 0, 0]
    tau_i_x = Ji[..., 0, 1]
    xi_i_t = Ji[..., 1, 0]
    xi_i_x = Ji[..., 1, 1]

    tau_j_t = Jj[..., 0, 0]
    tau_j_x = Jj[..., 0, 1]
    xi_j_t = Jj[..., 1, 0]
    xi_j_x = Jj[..., 1, 1]

    bracket_tau = (
        tau_i * tau_j_t
        + xi_i * tau_j_x
        - tau_j * tau_i_t
        - xi_j * tau_i_x
    )

    bracket_xi = (
        tau_i * xi_j_t
        + xi_i * xi_j_x
        - tau_j * xi_i_t
        - xi_j * xi_i_x
    )

    brackets = jnp.stack([bracket_tau, bracket_xi], axis=-1)  # (S, P, 2)

    # V has columns equal to flattened generator fields.
    # V shape: (2S, m)
    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T

    # B has columns equal to flattened brackets.
    # B shape: (2S, P)
    B = jnp.transpose(brackets, (1, 0, 2)).reshape(P, -1).T

    G = V.T @ V
    reg = (CONFIG["CLOSURE_RCOND"] ** 2) * jnp.eye(m, dtype=G.dtype)
    rhs = V.T @ B

    C = jnp.linalg.solve(G + reg, rhs)   # shape (m, P)
    B_proj = V @ C
    residual = B - B_proj

    return jnp.mean(residual**2)


def gram_independence_loss(W, tx):
    """
    Function-space Gram orthonormality:
        G_ab = E[tau_a tau_b + xi_a xi_b]
    """
    X = eval_generators_batch(W, tx)  # (S, m, 2)
    S = X.shape[0]
    m = X.shape[1]

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T  # (2S, m)
    G = (V.T @ V) / (2.0 * S)

    I = jnp.eye(m, dtype=G.dtype)
    loss_ind = jnp.mean((G - I) ** 2)

    diag = jnp.diag(G)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, CONFIG["MIN_GENERATOR_NORM"] - diag) ** 2
    )

    return loss_ind, norm_hinge, G


def sym_discovery_objective(W, sde_params, tx):
    det, proj = determining_loss(sde_params, W, tx)
    close = lie_closure_loss_global(W, tx)
    ind, norm_hinge, _ = gram_independence_loss(W, tx)

    total = (
        CONFIG["LAMBDA_DET"] * det
        + CONFIG["LAMBDA_CLOSE"] * close
        + CONFIG["LAMBDA_IND"] * ind
        + CONFIG["LAMBDA_PROJECTABLE"] * proj
        + CONFIG["LAMBDA_NORM_HINGE"] * norm_hinge
    )

    aux = {
        "total": total,
        "det": det,
        "close": close,
        "ind": ind,
        "projectable": proj,
        "norm_hinge": norm_hinge,
    }
    return total, aux


sym_value_and_grad = jax.value_and_grad(
    lambda W, sde_params, tx: sym_discovery_objective(W, sde_params, tx)[0]
)


@jax.jit
def sym_train_step(W, opt_state, sde_params, tx):
    loss, grads = sym_value_and_grad(W, sde_params, tx)
    W, opt_state = adam_update(
        W,
        grads,
        opt_state,
        CONFIG["SYM_DISCOVERY_LR"],
    )
    _, aux = sym_discovery_objective(W, sde_params, tx)
    return W, opt_state, aux


# ----------------------------
# Train symmetry basis weights W
# ----------------------------
W_opt_state = adam_init(W)
sym_history = []

t_start = time.time()

for epoch in range(1, CONFIG["SYM_DISCOVERY_EPOCHS"] + 1):
    key, subkey = jax.random.split(key)
    tx_disc = sample_tx_box(
        subkey,
        CONFIG["SYM_DISCOVERY_POINTS"],
        CONFIG["SYM_DISCOVERY_T_BOX"],
        CONFIG["SYM_DISCOVERY_X_BOX"],
    )

    W, W_opt_state, aux = sym_train_step(W, W_opt_state, sde_params, tx_disc)

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        aux_np = {k: float(v) for k, v in aux.items()}
        sym_history.append(aux_np)
        print(
            f"[SymDiscovery] epoch {epoch:5d} | "
            f"total {aux_np['total']: .4e} | "
            f"det {aux_np['det']: .4e} | "
            f"close {aux_np['close']: .4e} | "
            f"ind {aux_np['ind']: .4e} | "
            f"proj {aux_np['projectable']: .4e}"
        )

print(f"Symmetry discovery time: {time.time() - t_start:.2f} sec")

# Inspect learned coefficients
W_np = np.array(W)
print("\nLearned W shape:", W_np.shape)

for a in range(CONFIG["N_GENERATORS"]):
    print(f"\nGenerator X_{a+1}: top basis coefficients")
    order = np.argsort(-np.abs(W_np[a]))
    for idx in order[:min(8, N_BASIS)]:
        print(f"  {idx:02d} | coeff {W_np[a, idx]: .5f} | {BASIS_NAMES[idx]}")

# Inspect Gram matrix
key, subkey = jax.random.split(key)
tx_eval_sym = sample_tx_box(
    subkey,
    1024,
    CONFIG["SYM_DISCOVERY_T_BOX"],
    CONFIG["SYM_DISCOVERY_X_BOX"],
)
_, _, G_eval = gram_independence_loss(W, tx_eval_sym)
print("\nGenerator Gram matrix on eval collocation points:")
print(np.array(G_eval))

## Cell 6 — Stage F: symmetry-regularized surrogate refinement

In [ ]:
# ============================================================
# Cell 6 — Stage F: symmetry-regularized surrogate refinement
# ============================================================

def stage_f_objective(sde_params, W_fixed, batch, tx_extra):
    """
    Stage F objective.

    Uses ALL learned generators contained in W_fixed.

    Symmetry loss is evaluated on:
        1. trajectory batch points (t_k, x_k)
        2. extra collocation points sampled from user-defined tx box

    This matches the fact that the determining-equation residual is defined
    on the manifold, not only on observed trajectory points.
    """
    data_loss = data_nll_loss(sde_params, batch)

    t_batch, x_batch, _, _ = batch
    tx_data = jnp.stack([t_batch, x_batch], axis=1)
    tx_sym = jnp.concatenate([tx_data, tx_extra], axis=0)

    det, proj = determining_loss(sde_params, W_fixed, tx_sym)

    total = (
        data_loss
        + CONFIG["STAGE_F_LAMBDA_SYM_DET"] * det
        + CONFIG["STAGE_F_LAMBDA_PROJECTABLE"] * proj
    )

    aux = {
        "total": total,
        "data": data_loss,
        "sym_det": det,
        "projectable": proj,
    }
    return total, aux


stage_f_value_and_grad = jax.value_and_grad(
    lambda params, W_fixed, batch, tx_extra: stage_f_objective(
        params, W_fixed, batch, tx_extra
    )[0]
)


@jax.jit
def stage_f_train_step(sde_params, opt_state, W_fixed, batch, tx_extra):
    loss, grads = stage_f_value_and_grad(sde_params, W_fixed, batch, tx_extra)
    sde_params, opt_state = adam_update(
        sde_params,
        grads,
        opt_state,
        CONFIG["STAGE_F_LR"],
    )
    _, aux = stage_f_objective(sde_params, W_fixed, batch, tx_extra)
    return sde_params, opt_state, aux


# Save baseline params for comparison
sde_params_baseline = sde_params

# Start Stage F from baseline params
sde_params_refined = sde_params
stage_f_opt_state = adam_init(sde_params_refined)

stage_f_history = []
t_start = time.time()

for epoch in range(1, CONFIG["STAGE_F_EPOCHS"] + 1):
    key, key_batch, key_extra = jax.random.split(key, 3)

    batch = sample_batch(
        key_batch,
        train_data,
        CONFIG["STAGE_F_BATCH_SIZE"],
    )

    tx_extra = sample_tx_box(
        key_extra,
        CONFIG["STAGE_F_N_EXTRA_SYM_POINTS"],
        CONFIG["STAGE_F_EXTRA_T_BOX"],
        CONFIG["STAGE_F_EXTRA_X_BOX"],
    )

    sde_params_refined, stage_f_opt_state, aux = stage_f_train_step(
        sde_params_refined,
        stage_f_opt_state,
        W,
        batch,
        tx_extra,
    )

    if epoch % CONFIG["PRINT_EVERY"] == 0 or epoch == 1:
        aux_np = {k: float(v) for k, v in aux.items()}
        stage_f_history.append(aux_np)

        train_nll = float(data_nll_loss(sde_params_refined, train_data))
        test_nll = float(data_nll_loss(sde_params_refined, test_data))

        print(
            f"[Stage F] epoch {epoch:5d} | "
            f"total {aux_np['total']: .5f} | "
            f"data {aux_np['data']: .5f} | "
            f"sym_det {aux_np['sym_det']: .4e} | "
            f"train nll {train_nll: .5f} | "
            f"test nll {test_nll: .5f}"
        )

print(f"Stage F training time: {time.time() - t_start:.2f} sec")

# Final losses
baseline_train_nll = float(data_nll_loss(sde_params_baseline, train_data))
baseline_test_nll = float(data_nll_loss(sde_params_baseline, test_data))
refined_train_nll = float(data_nll_loss(sde_params_refined, train_data))
refined_test_nll = float(data_nll_loss(sde_params_refined, test_data))

print("\n================ Final NLL Comparison ================")
print(f"Baseline train NLL: {baseline_train_nll: .5f}")
print(f"Baseline test  NLL: {baseline_test_nll: .5f}")
print(f"Refined  train NLL: {refined_train_nll: .5f}")
print(f"Refined  test  NLL: {refined_test_nll: .5f}")

key, subkey = jax.random.split(key)
tx_sym_eval = sample_tx_box(
    subkey,
    2048,
    CONFIG["STAGE_F_EXTRA_T_BOX"],
    CONFIG["STAGE_F_EXTRA_X_BOX"],
)

det_base, proj_base = determining_loss(sde_params_baseline, W, tx_sym_eval)
det_ref, proj_ref = determining_loss(sde_params_refined, W, tx_sym_eval)

print("\n============ Symmetry Residual Comparison ============")
print(f"Baseline det residual: {float(det_base): .4e}")
print(f"Refined  det residual: {float(det_ref): .4e}")
print(f"Baseline projectability residual: {float(proj_base): .4e}")
print(f"Refined  projectability residual: {float(proj_ref): .4e}")

## Cell 6B — Paired roll-out visualizations under identical noise

In [ ]:
# ============================================================
# Cell 6B — Paired roll-out visualizations under identical noise
# ============================================================

# ------------------------------------------------------------
# User-editable visualization knobs
# ------------------------------------------------------------
# Change these values directly before running this cell.

ROLLOUT_N_TRAJECTORIES_TO_SHOW = 10           # number of trajectories drawn in each plot
ROLLOUT_T_BOX = (0.0, 2.0)                   # time interval shown and simulated
ROLLOUT_X_BOX = (-2.5, 2.5)                  # vertical plotting limits
ROLLOUT_X0_BOX = (-1.0, 1.0)                 # initial-state sampling box
ROLLOUT_SEED = CONFIG.get("ROLLOUT_VIS_SEED", 2025)

# File names for this cell's own saved outputs.
ROLLOUT_FIGURE_DPI = 250
ROLLOUT_SAVE_PDF = True

# Save directly to the Colab root/current working directory.
# This is the same level as folders like sde_surrogate_results_20260506_...
ROLLOUT_OUTPUT_DIR = Path.cwd()


def simulate_true_rollout_from_noise(x0, eps, t0, dt):
    """
    Ground-truth SDE roll-out using pre-sampled initial states and noise.
    """
    n_steps = eps.shape[0]
    times = t0 + jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, noise = inp
        f = true_drift(t, x)
        s = true_sigma(t, x)
        x_next = x + f * dt + s * sqrt_dt * noise
        return x_next, x_next

    _, xs_tail = jax.lax.scan(step, x0, (times[:-1], eps))
    paths = jnp.concatenate([x0[None, :], xs_tail], axis=0).T
    return times, paths


def simulate_surrogate_rollout_from_noise(params, x0, eps, t0, dt):
    """
    Neural SDE surrogate roll-out using pre-sampled initial states and noise.
    """
    n_steps = eps.shape[0]
    times = t0 + jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, noise = inp
        f, s = jax.vmap(lambda xx: surrogate_f_sigma(params, t, xx))(x)
        x_next = x + f * dt + s * sqrt_dt * noise
        return x_next, x_next

    _, xs_tail = jax.lax.scan(step, x0, (times[:-1], eps))
    paths = jnp.concatenate([x0[None, :], xs_tail], axis=0).T
    return times, paths


def rollout_summary_stats(paths):
    """
    Simple numerical summaries for saved diagnostics.
    """
    final_values = paths[:, -1]
    return {
        "final_mean": float(jnp.mean(final_values)),
        "final_std": float(jnp.std(final_values)),
        "final_min": float(jnp.min(final_values)),
        "final_max": float(jnp.max(final_values)),
        "path_min": float(jnp.min(paths)),
        "path_max": float(jnp.max(paths)),
    }


def save_rollout_figure(fig, stem):
    """
    Save this cell's figures directly in the Colab root/current working directory,
    not inside FIGURES_DIR.
    """
    ROLLOUT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    png_path = ROLLOUT_OUTPUT_DIR / f"{stem}.png"
    fig.savefig(png_path, dpi=ROLLOUT_FIGURE_DPI, bbox_inches="tight")
    print(f"[Saved rollout figure] {png_path}")

    if ROLLOUT_SAVE_PDF:
        pdf_path = ROLLOUT_OUTPUT_DIR / f"{stem}.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"[Saved rollout figure] {pdf_path}")


def trajectory_color(i, n):
    """
    Deterministic color assignment:
    trajectory i has the same color for ground truth and surrogate.
    """
    cmap = plt.get_cmap("tab10" if n <= 10 else "turbo")
    if n <= 1:
        return cmap(0)
    return cmap(i / max(n - 1, 1))


def plot_truth_vs_model_rollouts(
    times_np,
    truth_paths_np,
    model_paths_np,
    model_label,
    title,
    stem,
):
    """
    Plot ground-truth and one surrogate in the same axes.

    The i-th ground-truth path and i-th surrogate path use:
        - the same initial condition,
        - the same Brownian noise realization,
        - the same color.

    Line style distinguishes model type:
        - solid line: ground truth
        - dashed line: surrogate
    """
    fig, ax = plt.subplots(figsize=(9, 5))

    for i in range(n_paths_vis):
        color_i = trajectory_color(i, n_paths_vis)

        ax.plot(
            times_np,
            truth_paths_np[i],
            color=color_i,
            alpha=0.90,
            linewidth=2.0,
            linestyle="-",
            label="Ground Truth" if i == 0 else None,
        )

        ax.plot(
            times_np,
            model_paths_np[i],
            color=color_i,
            alpha=0.90,
            linewidth=1.8,
            linestyle="--",
            label=model_label if i == 0 else None,
        )

    ax.set_title(title)
    ax.set_xlabel("t")
    ax.set_ylabel("X_t")
    ax.set_xlim(float(t_start_vis), float(t_end_vis))
    ax.set_ylim(float(x_min_vis), float(x_max_vis))
    ax.grid(True)
    ax.legend()

    fig.tight_layout()
    save_rollout_figure(fig, stem)
    plt.show()


# ------------------------------------------------------------
# Validate user-editable inputs
# ------------------------------------------------------------

if ROLLOUT_N_TRAJECTORIES_TO_SHOW <= 0:
    raise ValueError("ROLLOUT_N_TRAJECTORIES_TO_SHOW must be positive.")

t_start_vis, t_end_vis = ROLLOUT_T_BOX
x_min_vis, x_max_vis = ROLLOUT_X_BOX
x0_min_vis, x0_max_vis = ROLLOUT_X0_BOX

if not (t_end_vis > t_start_vis):
    raise ValueError("ROLLOUT_T_BOX must satisfy end > start.")

if not (x_max_vis > x_min_vis):
    raise ValueError("ROLLOUT_X_BOX must satisfy max > min.")

if not (x0_max_vis > x0_min_vis):
    raise ValueError("ROLLOUT_X0_BOX must satisfy max > min.")


# ------------------------------------------------------------
# Shared initial states and shared Brownian noise
# ------------------------------------------------------------

rollout_key = jax.random.PRNGKey(int(ROLLOUT_SEED))
key_x0, key_noise = jax.random.split(rollout_key)

dt_vis = CONFIG["DT"]
n_steps_vis = int(round((t_end_vis - t_start_vis) / dt_vis))
n_paths_vis = int(ROLLOUT_N_TRAJECTORIES_TO_SHOW)

x0_vis = jax.random.uniform(
    key_x0,
    shape=(n_paths_vis,),
    minval=float(x0_min_vis),
    maxval=float(x0_max_vis),
)

# Same noise realizations for all three models.
eps_vis = jax.random.normal(
    key_noise,
    shape=(n_steps_vis, n_paths_vis),
)

times_true_vis, paths_true_vis = simulate_true_rollout_from_noise(
    x0_vis,
    eps_vis,
    float(t_start_vis),
    dt_vis,
)

_, paths_base_vis = simulate_surrogate_rollout_from_noise(
    sde_params_baseline,
    x0_vis,
    eps_vis,
    float(t_start_vis),
    dt_vis,
)

_, paths_ref_vis = simulate_surrogate_rollout_from_noise(
    sde_params_refined,
    x0_vis,
    eps_vis,
    float(t_start_vis),
    dt_vis,
)


# ------------------------------------------------------------
# Numerical roll-out comparison
# ------------------------------------------------------------

rollout_stats = {
    "config": {
        "n_trajectories_shown": int(n_paths_vis),
        "dt": float(dt_vis),
        "t_box": tuple(float(v) for v in ROLLOUT_T_BOX),
        "x_box": tuple(float(v) for v in ROLLOUT_X_BOX),
        "x0_box": tuple(float(v) for v in ROLLOUT_X0_BOX),
        "seed": int(ROLLOUT_SEED),
        "figure_output_dir": str(ROLLOUT_OUTPUT_DIR),
    },
    "ground_truth": rollout_summary_stats(paths_true_vis),
    "baseline_vanilla": rollout_summary_stats(paths_base_vis),
    "symmetry_regularized": rollout_summary_stats(paths_ref_vis),
    "pathwise_rmse_vs_truth": {
        "baseline_vanilla": float(rmse(paths_base_vis, paths_true_vis)),
        "symmetry_regularized": float(rmse(paths_ref_vis, paths_true_vis)),
    },
    "terminal_rmse_vs_truth": {
        "baseline_vanilla": float(rmse(paths_base_vis[:, -1], paths_true_vis[:, -1])),
        "symmetry_regularized": float(rmse(paths_ref_vis[:, -1], paths_true_vis[:, -1])),
    },
    "terminal_mae_vs_truth": {
        "baseline_vanilla": float(mae(paths_base_vis[:, -1], paths_true_vis[:, -1])),
        "symmetry_regularized": float(mae(paths_ref_vis[:, -1], paths_true_vis[:, -1])),
    },
}

TABLES_DIR.mkdir(parents=True, exist_ok=True)
rollout_stats_path = TABLES_DIR / "paired_rollout_comparison_stats.json"

with open(rollout_stats_path, "w", encoding="utf-8") as f:
    json.dump(rollout_stats, f, indent=2)

print("\n" + "=" * 72)
print("PAIRED ROLL-OUT COMPARISON UNDER IDENTICAL NOISE")
print("=" * 72)
print(f"Roll-out t-box       : {ROLLOUT_T_BOX}")
print(f"Plot x-box           : {ROLLOUT_X_BOX}")
print(f"Initial-state x0 box : {ROLLOUT_X0_BOX}")
print(f"Trajectories shown   : {n_paths_vis}")
print(f"Figure output dir    : {ROLLOUT_OUTPUT_DIR}")
print(f"Saved roll-out stats : {rollout_stats_path}")

print("\nPathwise RMSE vs ground truth:")
print(f"  baseline vanilla     : {rollout_stats['pathwise_rmse_vs_truth']['baseline_vanilla']: .6e}")
print(f"  symmetry-regularized : {rollout_stats['pathwise_rmse_vs_truth']['symmetry_regularized']: .6e}")

print("\nTerminal RMSE vs ground truth:")
print(f"  baseline vanilla     : {rollout_stats['terminal_rmse_vs_truth']['baseline_vanilla']: .6e}")
print(f"  symmetry-regularized : {rollout_stats['terminal_rmse_vs_truth']['symmetry_regularized']: .6e}")

print("\nTerminal MAE vs ground truth:")
print(f"  baseline vanilla     : {rollout_stats['terminal_mae_vs_truth']['baseline_vanilla']: .6e}")
print(f"  symmetry-regularized : {rollout_stats['terminal_mae_vs_truth']['symmetry_regularized']: .6e}")


# ------------------------------------------------------------
# Two trajectory-by-trajectory visualizations
# ------------------------------------------------------------

times_np = np.array(times_true_vis)
paths_true_np = np.array(paths_true_vis)
paths_base_np = np.array(paths_base_vis)
paths_ref_np = np.array(paths_ref_vis)

plot_truth_vs_model_rollouts(
    times_np=times_np,
    truth_paths_np=paths_true_np,
    model_paths_np=paths_base_np,
    model_label="Vanilla",
    title="Ground Truth vs Vanilla",
    stem="Vis_4_GTvs.Vanilla",
)

plot_truth_vs_model_rollouts(
    times_np=times_np,
    truth_paths_np=paths_true_np,
    model_paths_np=paths_ref_np,
    model_label="Gaeta-Lie",
    title="Ground Truth vs Gaeta-Lie",
    stem="Vis_4_GTvs.GL",
)

## Cell 7 — Diagnostics, numerical comparisons, and plots

In [ ]:
# ============================================================
# Cell 7 — Diagnostics, numerical comparisons, and plots
# ============================================================

def batched_surrogate_values(params, t_values, x_values):
    """
    Evaluate f_theta and sigma_theta on paired arrays t_values, x_values.
    """
    f, s = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t_values, x_values)
    return f, s


def safe_percent_change(new, old, eps=1e-12):
    """
    Percent change from old to new.
    Negative means the metric decreased.
    """
    return 100.0 * (new - old) / (abs(old) + eps)


def safe_percent_reduction(new, old, eps=1e-12):
    """
    Percent reduction from old to new.
    Positive means the metric improved/decreased.
    """
    return 100.0 * (old - new) / (abs(old) + eps)

def one_step_prediction_diagnostics(params, dataset):
    """
    Compare one-step Euler-Maruyama increment predictions on a dataset.

    For each sample:
        dx ≈ f_theta(t,x) dt + sigma_theta(t,x) sqrt(dt) noise

    We report:
        - increment mean RMSE/MAE
        - normalized squared residual mean:
              E[(dx - f dt)^2 / (sigma^2 dt)]
          which should be close to 1 for a well-calibrated Gaussian one-step model.
    """
    t0, x0, dt, dx = dataset
    f, s = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)

    mean_pred = f * dt
    var_pred = s**2 * dt + 1e-8

    inc_rmse = rmse(mean_pred, dx)
    inc_mae = mae(mean_pred, dx)

    standardized_sq_residual = jnp.mean((dx - mean_pred) ** 2 / var_pred)
    avg_pred_std = jnp.mean(jnp.sqrt(var_pred))
    avg_abs_dx = jnp.mean(jnp.abs(dx))

    return {
        "increment_rmse": float(inc_rmse),
        "increment_mae": float(inc_mae),
        "standardized_sq_residual_mean": float(standardized_sq_residual),
        "avg_pred_step_std": float(avg_pred_std),
        "avg_abs_dx": float(avg_abs_dx),
    }


# ============================================================
# Numerical diagnostic grid over the tx-manifold
# ============================================================

n_t_diag = min(80, CONFIG["N_PLOT_GRID"])
n_x_diag = min(160, CONFIG["N_PLOT_GRID"])

t_diag = jnp.linspace(
    CONFIG["STAGE_F_EXTRA_T_BOX"][0],
    CONFIG["STAGE_F_EXTRA_T_BOX"][1],
    n_t_diag,
)

x_diag = jnp.linspace(
    CONFIG["STAGE_F_EXTRA_X_BOX"][0],
    CONFIG["STAGE_F_EXTRA_X_BOX"][1],
    n_x_diag,
)

TT, XX = jnp.meshgrid(t_diag, x_diag, indexing="ij")
tx_diag = jnp.stack([TT.reshape(-1), XX.reshape(-1)], axis=1)

t_flat = tx_diag[:, 0]
x_flat = tx_diag[:, 1]

f_true_grid = jax.vmap(lambda tt, xx: true_drift(tt, xx))(t_flat, x_flat)
s_true_grid = jax.vmap(lambda tt, xx: true_sigma(tt, xx))(t_flat, x_flat)

f_base_grid, s_base_grid = batched_surrogate_values(
    sde_params_baseline, t_flat, x_flat
)

f_ref_grid, s_ref_grid = batched_surrogate_values(
    sde_params_refined, t_flat, x_flat
)


# ----------------------------
# Drift/diffusion reconstruction diagnostics
# ----------------------------
drift_mse_base = float(mse(f_base_grid, f_true_grid))
drift_mse_ref = float(mse(f_ref_grid, f_true_grid))

drift_rmse_base = float(rmse(f_base_grid, f_true_grid))
drift_rmse_ref = float(rmse(f_ref_grid, f_true_grid))

drift_mae_base = float(mae(f_base_grid, f_true_grid))
drift_mae_ref = float(mae(f_ref_grid, f_true_grid))

drift_rel_l2_base = float(relative_l2_error(f_base_grid, f_true_grid))
drift_rel_l2_ref = float(relative_l2_error(f_ref_grid, f_true_grid))


sigma_mse_base = float(mse(s_base_grid, s_true_grid))
sigma_mse_ref = float(mse(s_ref_grid, s_true_grid))

sigma_rmse_base = float(rmse(s_base_grid, s_true_grid))
sigma_rmse_ref = float(rmse(s_ref_grid, s_true_grid))

sigma_mae_base = float(mae(s_base_grid, s_true_grid))
sigma_mae_ref = float(mae(s_ref_grid, s_true_grid))

sigma_rel_l2_base = float(relative_l2_error(s_base_grid, s_true_grid))
sigma_rel_l2_ref = float(relative_l2_error(s_ref_grid, s_true_grid))


# ----------------------------
# NLL diagnostics
# ----------------------------
baseline_train_nll = float(data_nll_loss(sde_params_baseline, train_data))
baseline_test_nll = float(data_nll_loss(sde_params_baseline, test_data))

refined_train_nll = float(data_nll_loss(sde_params_refined, train_data))
refined_test_nll = float(data_nll_loss(sde_params_refined, test_data))


# ----------------------------
# Symmetry residual diagnostics
# ----------------------------
key, subkey = jax.random.split(key)
tx_sym_eval = sample_tx_box(
    subkey,
    4096,
    CONFIG["STAGE_F_EXTRA_T_BOX"],
    CONFIG["STAGE_F_EXTRA_X_BOX"],
)

det_base, proj_base = determining_loss(sde_params_baseline, W, tx_sym_eval)
det_ref, proj_ref = determining_loss(sde_params_refined, W, tx_sym_eval)

det_base = float(det_base)
det_ref = float(det_ref)

proj_base = float(proj_base)
proj_ref = float(proj_ref)


# ----------------------------
# One-step prediction diagnostics
# ----------------------------
train_step_base = one_step_prediction_diagnostics(sde_params_baseline, train_data)
train_step_ref = one_step_prediction_diagnostics(sde_params_refined, train_data)

test_step_base = one_step_prediction_diagnostics(sde_params_baseline, test_data)
test_step_ref = one_step_prediction_diagnostics(sde_params_refined, test_data)


# ============================================================
# Print numerical diagnostics
# ============================================================

print("\n" + "=" * 72)
print("NUMERICAL COMPARISON: BASELINE vs REFINED")
print("=" * 72)

print("\n[1] Negative log-likelihood")
print(f"Train NLL baseline : {baseline_train_nll: .6f}")
print(f"Train NLL refined  : {refined_train_nll: .6f}")
print(f"Train NLL change   : {refined_train_nll - baseline_train_nll: .6f}")
print(f"Train NLL % change : {safe_percent_change(refined_train_nll, baseline_train_nll): .2f}%")

print(f"\nTest NLL baseline  : {baseline_test_nll: .6f}")
print(f"Test NLL refined   : {refined_test_nll: .6f}")
print(f"Test NLL change    : {refined_test_nll - baseline_test_nll: .6f}")
print(f"Test NLL % change  : {safe_percent_change(refined_test_nll, baseline_test_nll): .2f}%")


print("\n[2] Drift reconstruction on tx-grid")
print(f"Drift MSE baseline       : {drift_mse_base: .6e}")
print(f"Drift MSE refined        : {drift_mse_ref: .6e}")
print(f"Drift MSE reduction      : {safe_percent_reduction(drift_mse_ref, drift_mse_base): .2f}%")

print(f"\nDrift RMSE baseline      : {drift_rmse_base: .6e}")
print(f"Drift RMSE refined       : {drift_rmse_ref: .6e}")
print(f"Drift RMSE reduction     : {safe_percent_reduction(drift_rmse_ref, drift_rmse_base): .2f}%")

print(f"\nDrift MAE baseline       : {drift_mae_base: .6e}")
print(f"Drift MAE refined        : {drift_mae_ref: .6e}")
print(f"Drift MAE reduction      : {safe_percent_reduction(drift_mae_ref, drift_mae_base): .2f}%")

print(f"\nDrift relative L2 baseline: {drift_rel_l2_base: .6e}")
print(f"Drift relative L2 refined : {drift_rel_l2_ref: .6e}")
print(f"Drift relative L2 reduction: {safe_percent_reduction(drift_rel_l2_ref, drift_rel_l2_base): .2f}%")


print("\n[3] Diffusion reconstruction on tx-grid")
print(f"Sigma MSE baseline       : {sigma_mse_base: .6e}")
print(f"Sigma MSE refined        : {sigma_mse_ref: .6e}")
print(f"Sigma MSE reduction      : {safe_percent_reduction(sigma_mse_ref, sigma_mse_base): .2f}%")

print(f"\nSigma RMSE baseline      : {sigma_rmse_base: .6e}")
print(f"Sigma RMSE refined       : {sigma_rmse_ref: .6e}")
print(f"Sigma RMSE reduction     : {safe_percent_reduction(sigma_rmse_ref, sigma_rmse_base): .2f}%")

print(f"\nSigma MAE baseline       : {sigma_mae_base: .6e}")
print(f"Sigma MAE refined        : {sigma_mae_ref: .6e}")
print(f"Sigma MAE reduction      : {safe_percent_reduction(sigma_mae_ref, sigma_mae_base): .2f}%")

print(f"\nSigma relative L2 baseline: {sigma_rel_l2_base: .6e}")
print(f"Sigma relative L2 refined : {sigma_rel_l2_ref: .6e}")
print(f"Sigma relative L2 reduction: {safe_percent_reduction(sigma_rel_l2_ref, sigma_rel_l2_base): .2f}%")


print("\n[4] Symmetry determining-equation residual")
print(f"Baseline determining residual : {det_base: .6e}")
print(f"Refined determining residual  : {det_ref: .6e}")
print(f"Residual reduction            : {safe_percent_reduction(det_ref, det_base): .2f}%")

print(f"\nBaseline projectability residual : {proj_base: .6e}")
print(f"Refined projectability residual  : {proj_ref: .6e}")
print(f"Projectability residual reduction: {safe_percent_reduction(proj_ref, proj_base): .2f}%")


print("\n[5] One-step increment prediction diagnostics: train")
print(f"Increment RMSE baseline : {train_step_base['increment_rmse']: .6e}")
print(f"Increment RMSE refined  : {train_step_ref['increment_rmse']: .6e}")
print(f"RMSE reduction          : {safe_percent_reduction(train_step_ref['increment_rmse'], train_step_base['increment_rmse']): .2f}%")

print(f"\nIncrement MAE baseline  : {train_step_base['increment_mae']: .6e}")
print(f"Increment MAE refined   : {train_step_ref['increment_mae']: .6e}")
print(f"MAE reduction           : {safe_percent_reduction(train_step_ref['increment_mae'], train_step_base['increment_mae']): .2f}%")

print(f"\nMean standardized squared residual baseline : {train_step_base['standardized_sq_residual_mean']: .6f}")
print(f"Mean standardized squared residual refined  : {train_step_ref['standardized_sq_residual_mean']: .6f}")
print("Ideal calibration value is approximately 1.0.")


print("\n[6] One-step increment prediction diagnostics: test")
print(f"Increment RMSE baseline : {test_step_base['increment_rmse']: .6e}")
print(f"Increment RMSE refined  : {test_step_ref['increment_rmse']: .6e}")
print(f"RMSE reduction          : {safe_percent_reduction(test_step_ref['increment_rmse'], test_step_base['increment_rmse']): .2f}%")

print(f"\nIncrement MAE baseline  : {test_step_base['increment_mae']: .6e}")
print(f"Increment MAE refined   : {test_step_ref['increment_mae']: .6e}")
print(f"MAE reduction           : {safe_percent_reduction(test_step_ref['increment_mae'], test_step_base['increment_mae']): .2f}%")

print(f"\nMean standardized squared residual baseline : {test_step_base['standardized_sq_residual_mean']: .6f}")
print(f"Mean standardized squared residual refined  : {test_step_ref['standardized_sq_residual_mean']: .6f}")
print("Ideal calibration value is approximately 1.0.")


# ============================================================
# Plot training curves
# ============================================================

plt.figure(figsize=(8, 4))
plt.plot(baseline_losses)
plt.xlabel("baseline epoch")
plt.ylabel("batch NLL")
plt.title("Baseline neural SDE training")
plt.grid(True)
plt.show()


if len(sym_history) > 0:
    plt.figure(figsize=(8, 4))
    plt.plot([h["det"] for h in sym_history], label="determining")
    plt.plot([h["close"] for h in sym_history], label="closure")
    plt.plot([h["ind"] for h in sym_history], label="independence")
    plt.yscale("log")
    plt.xlabel(f"checkpoint every {CONFIG['PRINT_EVERY']} epochs")
    plt.ylabel("loss")
    plt.title("Symmetry discovery losses")
    plt.legend()
    plt.grid(True)
    plt.show()


if len(stage_f_history) > 0:
    plt.figure(figsize=(8, 4))
    plt.plot([h["data"] for h in stage_f_history], label="data NLL")
    plt.plot([h["sym_det"] for h in stage_f_history], label="sym det")
    plt.yscale("log")
    plt.xlabel(f"checkpoint every {CONFIG['PRINT_EVERY']} epochs")
    plt.ylabel("loss")
    plt.title("Stage F losses")
    plt.legend()
    plt.grid(True)
    plt.show()


# ============================================================
# Compare true vs learned drift/diffusion at selected time slices
# ============================================================

x_grid = jnp.linspace(
    CONFIG["STAGE_F_EXTRA_X_BOX"][0],
    CONFIG["STAGE_F_EXTRA_X_BOX"][1],
    CONFIG["N_PLOT_GRID"],
)

t_slices = [0.0, 0.5 * CONFIG["T_FINAL"], CONFIG["T_FINAL"]]

for t0 in t_slices:
    t_grid = jnp.full_like(x_grid, t0)

    f_true = jax.vmap(lambda xx: true_drift(t0, xx))(x_grid)
    s_true = jax.vmap(lambda xx: true_sigma(t0, xx))(x_grid)

    f_base, s_base = batched_surrogate_values(sde_params_baseline, t_grid, x_grid)
    f_ref, s_ref = batched_surrogate_values(sde_params_refined, t_grid, x_grid)

    drift_slice_rmse_base = float(rmse(f_base, f_true))
    drift_slice_rmse_ref = float(rmse(f_ref, f_true))

    sigma_slice_rmse_base = float(rmse(s_base, s_true))
    sigma_slice_rmse_ref = float(rmse(s_ref, s_true))

    print("\n" + "-" * 72)
    print(f"Slice diagnostics at t = {t0:.3f}")
    print(f"Drift RMSE baseline : {drift_slice_rmse_base: .6e}")
    print(f"Drift RMSE refined  : {drift_slice_rmse_ref: .6e}")
    print(f"Drift RMSE reduction: {safe_percent_reduction(drift_slice_rmse_ref, drift_slice_rmse_base): .2f}%")
    print(f"Sigma RMSE baseline : {sigma_slice_rmse_base: .6e}")
    print(f"Sigma RMSE refined  : {sigma_slice_rmse_ref: .6e}")
    print(f"Sigma RMSE reduction: {safe_percent_reduction(sigma_slice_rmse_ref, sigma_slice_rmse_base): .2f}%")

    plt.figure(figsize=(8, 4))
    plt.plot(np.array(x_grid), np.array(f_true), label="true drift")
    plt.plot(np.array(x_grid), np.array(f_base), "--", label="baseline drift")
    plt.plot(np.array(x_grid), np.array(f_ref), ":", label="refined drift")
    plt.xlabel("x")
    plt.ylabel("f(t,x)")
    plt.title(f"Drift comparison at t={t0:.2f}")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.plot(np.array(x_grid), np.array(s_true), label="true sigma")
    plt.plot(np.array(x_grid), np.array(s_base), "--", label="baseline sigma")
    plt.plot(np.array(x_grid), np.array(s_ref), ":", label="refined sigma")
    plt.xlabel("x")
    plt.ylabel("sigma(t,x)")
    plt.title(f"Diffusion comparison at t={t0:.2f}")
    plt.legend()
    plt.grid(True)
    plt.show()


# ============================================================
# Bar plots: baseline vs refined metrics
# ============================================================

metric_names = [
    "Test NLL",
    "Drift RMSE",
    "Sigma RMSE",
    "Sym Det",
    "Test Inc RMSE",
]

baseline_metric_values = np.array([
    baseline_test_nll,
    drift_rmse_base,
    sigma_rmse_base,
    det_base,
    test_step_base["increment_rmse"],
], dtype=float)

refined_metric_values = np.array([
    refined_test_nll,
    drift_rmse_ref,
    sigma_rmse_ref,
    det_ref,
    test_step_ref["increment_rmse"],
], dtype=float)

x_pos = np.arange(len(metric_names))
width = 0.35

plt.figure(figsize=(10, 4))
plt.bar(x_pos - width / 2, baseline_metric_values, width, label="baseline")
plt.bar(x_pos + width / 2, refined_metric_values, width, label="refined")
plt.xticks(x_pos, metric_names, rotation=25, ha="right")
plt.yscale("log")
plt.ylabel("metric value, log scale")
plt.title("Baseline vs refined numerical diagnostics")
plt.legend()
plt.grid(True, axis="y")
plt.tight_layout()
plt.show()


# ============================================================
# Visualize learned generators
# ============================================================

key, subkey = jax.random.split(key)
tx_vis = sample_tx_box(
    subkey,
    2000,
    CONFIG["SYM_DISCOVERY_T_BOX"],
    CONFIG["SYM_DISCOVERY_X_BOX"],
)

X_vis = eval_generators_batch(W, tx_vis)  # shape (S, m, 2)

tx_vis_np = np.array(tx_vis)
X_vis_np = np.array(X_vis)

for a in range(CONFIG["N_GENERATORS"]):
    plt.figure(figsize=(7, 5))
    plt.quiver(
        tx_vis_np[:, 0],
        tx_vis_np[:, 1],
        X_vis_np[:, a, 0],
        X_vis_np[:, a, 1],
        angles="xy",
        scale_units="xy",
        scale=30,
        width=0.002,
        alpha=0.6,
    )
    plt.xlabel("t")
    plt.ylabel("x")
    plt.title(f"Learned generator X_{a+1}: tau(t)∂t + xi(t,x)∂x")
    plt.grid(True)
    plt.show()

# ============================================================
# Save numerical diagnostics as structured files
# ============================================================

numerical_results = {
    "config": {
        "N_GENERATORS": int(CONFIG["N_GENERATORS"]),
        "N_BASIS": int(N_BASIS),
        "STAGE_F_N_EXTRA_SYM_POINTS": int(CONFIG["STAGE_F_N_EXTRA_SYM_POINTS"]),
        "STAGE_F_EXTRA_T_BOX": tuple(float(v) for v in CONFIG["STAGE_F_EXTRA_T_BOX"]),
        "STAGE_F_EXTRA_X_BOX": tuple(float(v) for v in CONFIG["STAGE_F_EXTRA_X_BOX"]),
    },
    "nll": {
        "baseline_train": float(baseline_train_nll),
        "baseline_test": float(baseline_test_nll),
        "refined_train": float(refined_train_nll),
        "refined_test": float(refined_test_nll),
        "train_change": float(refined_train_nll - baseline_train_nll),
        "test_change": float(refined_test_nll - baseline_test_nll),
        "train_percent_change": float(safe_percent_change(refined_train_nll, baseline_train_nll)),
        "test_percent_change": float(safe_percent_change(refined_test_nll, baseline_test_nll)),
    },
    "drift_reconstruction": {
        "mse_baseline": float(drift_mse_base),
        "mse_refined": float(drift_mse_ref),
        "mse_reduction_percent": float(safe_percent_reduction(drift_mse_ref, drift_mse_base)),
        "rmse_baseline": float(drift_rmse_base),
        "rmse_refined": float(drift_rmse_ref),
        "rmse_reduction_percent": float(safe_percent_reduction(drift_rmse_ref, drift_rmse_base)),
        "mae_baseline": float(drift_mae_base),
        "mae_refined": float(drift_mae_ref),
        "mae_reduction_percent": float(safe_percent_reduction(drift_mae_ref, drift_mae_base)),
        "relative_l2_baseline": float(drift_rel_l2_base),
        "relative_l2_refined": float(drift_rel_l2_ref),
        "relative_l2_reduction_percent": float(safe_percent_reduction(drift_rel_l2_ref, drift_rel_l2_base)),
    },
    "sigma_reconstruction": {
        "mse_baseline": float(sigma_mse_base),
        "mse_refined": float(sigma_mse_ref),
        "mse_reduction_percent": float(safe_percent_reduction(sigma_mse_ref, sigma_mse_base)),
        "rmse_baseline": float(sigma_rmse_base),
        "rmse_refined": float(sigma_rmse_ref),
        "rmse_reduction_percent": float(safe_percent_reduction(sigma_rmse_ref, sigma_rmse_base)),
        "mae_baseline": float(sigma_mae_base),
        "mae_refined": float(sigma_mae_ref),
        "mae_reduction_percent": float(safe_percent_reduction(sigma_mae_ref, sigma_mae_base)),
        "relative_l2_baseline": float(sigma_rel_l2_base),
        "relative_l2_refined": float(sigma_rel_l2_ref),
        "relative_l2_reduction_percent": float(safe_percent_reduction(sigma_rel_l2_ref, sigma_rel_l2_base)),
    },
    "symmetry_residuals": {
        "det_baseline": float(det_base),
        "det_refined": float(det_ref),
        "det_reduction_percent": float(safe_percent_reduction(det_ref, det_base)),
        "projectability_baseline": float(proj_base),
        "projectability_refined": float(proj_ref),
        "projectability_reduction_percent": float(safe_percent_reduction(proj_ref, proj_base)),
    },
    "one_step_train": {
        "baseline": train_step_base,
        "refined": train_step_ref,
        "increment_rmse_reduction_percent": float(
            safe_percent_reduction(
                train_step_ref["increment_rmse"],
                train_step_base["increment_rmse"],
            )
        ),
        "increment_mae_reduction_percent": float(
            safe_percent_reduction(
                train_step_ref["increment_mae"],
                train_step_base["increment_mae"],
            )
        ),
    },
    "one_step_test": {
        "baseline": test_step_base,
        "refined": test_step_ref,
        "increment_rmse_reduction_percent": float(
            safe_percent_reduction(
                test_step_ref["increment_rmse"],
                test_step_base["increment_rmse"],
            )
        ),
        "increment_mae_reduction_percent": float(
            safe_percent_reduction(
                test_step_ref["increment_mae"],
                test_step_base["increment_mae"],
            )
        ),
    },
    "top_generator_coefficients": {},
}

for a in range(CONFIG["N_GENERATORS"]):
    order = np.argsort(-np.abs(W_np[a]))
    numerical_results["top_generator_coefficients"][f"X_{a+1}"] = [
        {
            "basis_index": int(idx),
            "coefficient": float(W_np[a, idx]),
            "basis_name": BASIS_NAMES[idx],
        }
        for idx in order[:8]
    ]

numerical_results_json_path = TABLES_DIR / "numerical_results.json"
with open(numerical_results_json_path, "w", encoding="utf-8") as f:
    json.dump(numerical_results, f, indent=2)

print(f"\nSaved structured numerical results to: {numerical_results_json_path}")


# Also save a compact CSV table for the main scalar metrics.
main_metrics_rows = [
    ["train_nll", baseline_train_nll, refined_train_nll],
    ["test_nll", baseline_test_nll, refined_test_nll],
    ["drift_mse", drift_mse_base, drift_mse_ref],
    ["drift_rmse", drift_rmse_base, drift_rmse_ref],
    ["drift_mae", drift_mae_base, drift_mae_ref],
    ["drift_relative_l2", drift_rel_l2_base, drift_rel_l2_ref],
    ["sigma_mse", sigma_mse_base, sigma_mse_ref],
    ["sigma_rmse", sigma_rmse_base, sigma_rmse_ref],
    ["sigma_mae", sigma_mae_base, sigma_mae_ref],
    ["sigma_relative_l2", sigma_rel_l2_base, sigma_rel_l2_ref],
    ["symmetry_det_residual", det_base, det_ref],
    ["projectability_residual", proj_base, proj_ref],
    ["train_increment_rmse", train_step_base["increment_rmse"], train_step_ref["increment_rmse"]],
    ["train_increment_mae", train_step_base["increment_mae"], train_step_ref["increment_mae"]],
    ["test_increment_rmse", test_step_base["increment_rmse"], test_step_ref["increment_rmse"]],
    ["test_increment_mae", test_step_base["increment_mae"], test_step_ref["increment_mae"]],
]

main_metrics_csv_path = TABLES_DIR / "main_metrics_baseline_vs_refined.csv"

with open(main_metrics_csv_path, "w", encoding="utf-8") as f:
    f.write("metric,baseline,refined,absolute_change,percent_reduction\n")
    for name, base_val, ref_val in main_metrics_rows:
        abs_change = float(ref_val - base_val)
        pct_reduction = float(safe_percent_reduction(ref_val, base_val))
        f.write(
            f"{name},{float(base_val)},{float(ref_val)},{abs_change},{pct_reduction}\n"
        )

print(f"Saved compact metric CSV to: {main_metrics_csv_path}")

# ============================================================
# Print compact final report
# ============================================================

W_np = np.array(W)

print("\n" + "=" * 72)
print("COMPACT FINAL REPORT")
print("=" * 72)

print(f"N_GENERATORS used in Stage F: {CONFIG['N_GENERATORS']}")
print(f"N_BASIS in editable Gaeta library: {N_BASIS}")
print(f"Stage F extra symmetry points per step: {CONFIG['STAGE_F_N_EXTRA_SYM_POINTS']}")
print(f"Stage F extra t-box: {CONFIG['STAGE_F_EXTRA_T_BOX']}")
print(f"Stage F extra x-box: {CONFIG['STAGE_F_EXTRA_X_BOX']}")

print("\nNLL:")
print(f"  baseline train: {baseline_train_nll: .6f}")
print(f"  baseline test : {baseline_test_nll: .6f}")
print(f"  refined  train: {refined_train_nll: .6f}")
print(f"  refined  test : {refined_test_nll: .6f}")

print("\nDrift/Diffusion reconstruction over tx-grid:")
print(f"  drift RMSE baseline : {drift_rmse_base: .6e}")
print(f"  drift RMSE refined  : {drift_rmse_ref: .6e}")
print(f"  sigma RMSE baseline : {sigma_rmse_base: .6e}")
print(f"  sigma RMSE refined  : {sigma_rmse_ref: .6e}")

print("\nSymmetry determining residual:")
print(f"  baseline: {det_base: .6e}")
print(f"  refined : {det_ref: .6e}")

print("\nTop coefficients per learned generator:")
for a in range(CONFIG["N_GENERATORS"]):
    order = np.argsort(-np.abs(W_np[a]))
    print(f"\nX_{a+1}:")
    for idx in order[:5]:
        print(f"  coeff {W_np[a, idx]: .5f} | {BASIS_NAMES[idx]}")

## Final cell — Save everything into one zip file

In [ ]:
# ============================================================
# Final cell — Save everything into one zip file
# ============================================================

# Save any figures that might still be open and not yet saved.
_save_new_open_figures()

# Save the final CONFIG too.
config_path = RESULTS_DIR / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Saved CONFIG to: {config_path}")

# Flush the print log before zipping.
try:
    sys.stdout.flush()
    sys.stderr.flush()
    _log_file.flush()
except Exception:
    pass

ZIP_BASE = Path.cwd() / f"{RESULTS_DIR.name}"
ZIP_PATH = shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(RESULTS_DIR),
)

print("\n" + "=" * 72)
print("SAVED COMPLETE RESULTS ZIP")
print("=" * 72)
print(f"Zip file: {ZIP_PATH}")
print(f"Results directory: {RESULTS_DIR}")

# Optional Colab download helper.
try:
    from google.colab import files
    print("In Colab, you can download the zip using:")
    print(f"files.download('{ZIP_PATH}')")
except Exception:
    pass